<a href="https://colab.research.google.com/github/ddickson28/FPSO-BN/blob/Adding-R-node/FPSOBN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install mbnpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.3/105.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0,

In [1]:
#Import modules
import numpy as np #importing a module
from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy
import itertools
from scipy.stats import truncnorm
from math import erf, sqrt

In [2]:
#1 Define function for calculating combinations
def generate_indexed_combinations(state_counts):

    # Create ranges for each variable: 0..n_states-1
    ranges = [range(n) for n in state_counts]

    # Compute Cartesian product
    combos = list(itertools.product(*ranges))

    # Convert to numpy array
    return np.array(combos, dtype=int)


In [3]:
"""Defining the child node probability table requires defining all of the parent state
combinations and then using a T-Normal distribution to create discrete bins
across the defined child states|parent combination"""

# Define further node weights as required, ensure consistent ordering and sum to
# 1
N1 = 0.4  #PreviousRepair 2 states
N2 = 0.15 #OffloadCycle 4 states
N3 = 0.15 #LoadRange 4 states
N4 = 0.3  #Resistance 5 states

#Define the numeric scores for the states of ea. node. Follows ordering of nodes
#above.

P_Repair=np.array([1,0])
P_Offload=np.array([1, 0.66, 0.33, 0])
P_Load=np.array([1, 0.66, 0.33, 0])
P_Resistance=np.array([1, 0.75, 0.5, 0.25, 0])
#P_Additional=np.array([1, 0.75, 0.5, 0.25, 0])

#Define the weighted scores for the node vectors
A = N1*P_Repair
B = N2*P_Offload
C = N3*P_Load
D = N4*P_Resistance


#Create pairwise sum w/ specific ordering. Vector 1 moves slowest, vector 2
#moves fastest.

Parent_combination = A[:, None, None, None] + B[None, :, None, None] + C[None, None, :, None] + D[None, None, None, :]

print(Parent_combination)

#Takes two rows and flattens into a 1-D vector
mu = Parent_combination.reshape(-1)

print(mu)

#Calculating truncated normal from mu vector above and specficied variance

var = 0.15
std = np.sqrt(var)

#Define intervals for truncnorm 0, 0.5, 1.
bins = np.array([0.0, 0.5, 1])

#Build truncnorm and calculate CDF for each bin interval

a = (0 - mu) / std
b = (1 - mu) / std

cdf_val = truncnorm.cdf(bins[:, None], a, b, loc=mu, scale=std)

prob_vectors = np.diff(cdf_val, axis=0).T
prob_vectors = prob_vectors[:,::-1]
sum_vectors=prob_vectors.sum(axis=1)

print(prob_vectors)
print(sum_vectors)

#***Need to build the order here, 1st from each column***
flat_prob = prob_vectors.T.flatten()

print(flat_prob)


[[[[1.     0.925  0.85   0.775  0.7   ]
   [0.949  0.874  0.799  0.724  0.649 ]
   [0.8995 0.8245 0.7495 0.6745 0.5995]
   [0.85   0.775  0.7    0.625  0.55  ]]

  [[0.949  0.874  0.799  0.724  0.649 ]
   [0.898  0.823  0.748  0.673  0.598 ]
   [0.8485 0.7735 0.6985 0.6235 0.5485]
   [0.799  0.724  0.649  0.574  0.499 ]]

  [[0.8995 0.8245 0.7495 0.6745 0.5995]
   [0.8485 0.7735 0.6985 0.6235 0.5485]
   [0.799  0.724  0.649  0.574  0.499 ]
   [0.7495 0.6745 0.5995 0.5245 0.4495]]

  [[0.85   0.775  0.7    0.625  0.55  ]
   [0.799  0.724  0.649  0.574  0.499 ]
   [0.7495 0.6745 0.5995 0.5245 0.4495]
   [0.7    0.625  0.55   0.475  0.4   ]]]


 [[[0.6    0.525  0.45   0.375  0.3   ]
   [0.549  0.474  0.399  0.324  0.249 ]
   [0.4995 0.4245 0.3495 0.2745 0.1995]
   [0.45   0.375  0.3    0.225  0.15  ]]

  [[0.549  0.474  0.399  0.324  0.249 ]
   [0.498  0.423  0.348  0.273  0.198 ]
   [0.4485 0.3735 0.2985 0.2235 0.1485]
   [0.399  0.324  0.249  0.174  0.099 ]]

  [[0.4995 0.4245 0.3495 0

In [4]:
"Building the BN and relationship"
from mbnpy import variable, cpm, inference #importing relevant code classes from MBNpy

#1 Define the variables (nodes). N1, N2, ... ,Nn. Child node last


PreviousRepair = variable.Variable('PreviousRepair', ['True','False']) #has the feature cracked before
OffloadCycle = variable.Variable('OffloadCycles', ['Critical','High','Med','Low']) #No. of cycles
LoadRange = variable.Variable('LoadRange', ['Peak','High','Moderate', 'Low']) #Extent of load reversal at location of interest
Resistance = variable.Variable('Resistance', ['High','Med','Low']) #Calculated Resistance defined as normal distribution in 5 steps


CrackLocationInterest = variable.Variable('CrackLocationInterest', ['True', 'False'])


#2 Define the cpm of the variables from 1 in order specified above.

cpm_PreviousRepair = cpm.Cpm(
									[PreviousRepair], no_child=1,
									 C=np.array([[0],[1]], dtype=int),
									 p=np.array([0.5,0.5])
)

cpm_OffloadCycle = cpm.Cpm(
									[OffloadCycle], no_child=1,
									 C=np.array([[0],[1],[2],[3]], dtype=int),
									 p=np.array([[0.25],[0.25],[0.25],[0.25]])
)

cpm_LoadRange = cpm.Cpm(
									[LoadRange], no_child=1,
									 C=np.array([[0],[1],[2],[3]], dtype=int),
									 p=np.array([[0.25],[0.25],[0.25],[0.25]])
)

cpm_Resistance = cpm.Cpm(
									[Resistance], no_child=1,
									 C=np.array([[0],[1],[2],[3],[4]], dtype=int),
									 p=np.array([[0.001],[0.157],[0.683],[0.157],[0.001]]) #From Normal distribution mu 734, St. Dev 1
)


# Creates all possible child states given parents
state_counts = [2, 2, 4, 4, 5]  # Child: 3 states, Parent1: 2 states, Parent2: 3 states parent3: 4 states, parent4: 5 states
#Go to code section above for creating P_child flat

C_child= generate_indexed_combinations(state_counts)
print(C_child)

P_child=flat_prob
print(P_child)

cpm_CrackLocationInterest = cpm.Cpm(
									[CrackLocationInterest, PreviousRepair, OffloadCycle, LoadRange, Resistance], no_child=0,
									 C=C_child,
									 p=P_child
)

print(cpm_CrackLocationInterest)

[[0 0 0 0 0]
 [0 0 0 0 1]
 [0 0 0 0 2]
 ...
 [1 1 3 3 2]
 [1 1 3 3 3]
 [1 1 3 3 4]]
[0.81126366 0.77515376 0.73456658 0.68974675 0.64119675 0.78720105
 0.7480317  0.70452434 0.65709715 0.60643166 0.76184933 0.71978309
 0.6736253  0.62396843 0.57165926 0.73456658 0.68974675 0.64119675
 0.58967996 0.53618951 0.78720105 0.7480317  0.70452434 0.65709715
 0.60643166 0.76105066 0.71889849 0.67266403 0.62294502 0.570593
 0.73371043 0.68881018 0.64019253 0.58862585 0.53510733 0.70452434
 0.65709715 0.60643166 0.55345307 0.49927499 0.76184933 0.71978309
 0.6736253  0.62396843 0.57165926 0.73371043 0.68881018 0.64019253
 0.58862585 0.53510733 0.70452434 0.65709715 0.60643166 0.55345307
 0.49927499 0.6736253  0.62396843 0.57165926 0.51775565 0.46344983
 0.73456658 0.68974675 0.64119675 0.58967996 0.53618951 0.70452434
 0.65709715 0.60643166 0.55345307 0.49927499 0.6736253  0.62396843
 0.57165926 0.51775565 0.46344983 0.64119675 0.58967996 0.53618951
 0.48188229 0.42798546 0.57201454 0.51811771 0.

In [5]:
# Display code only. Displays combinations and asscoiated probability.

import sys
import numpy as np
import pandas as pd

# Set print options to display the full arrays without truncation
np.set_printoptions(threshold=sys.maxsize, linewidth=sys.maxsize)

print("Full combinations (C_child):")
print(C_child)

print("\nFull probabilities (P_child):")
print(P_child)

# Create column names for C_child
# C_child represents combinations of [Child_state, Parent1_state, Parent2_state, Parent3_state, Parent4_state]
parent_vars_list = ['PreviousRepair', 'OffloadCycle', 'LoadRange', 'Resistance']
child_var_name = 'CrackLocationInterest'
all_col_names = [child_var_name] + parent_vars_list

df_cpm = pd.DataFrame(C_child, columns=all_col_names)
# The P_child array contains the probabilities corresponding to the combinations in C_child.
# Each row of C_child (representing a specific combination of child and parent states)
# has an associated probability in P_child.
# So we can add P_child as a new column to df_cpm.
df_cpm['Probability'] = P_child

print("\nFull Conditional Probability Table (DataFrame):")
# Display the DataFrame, pandas should handle display for the full table
print(df_cpm.to_string())

Full combinations (C_child):
[[0 0 0 0 0]
 [0 0 0 0 1]
 [0 0 0 0 2]
 [0 0 0 0 3]
 [0 0 0 0 4]
 [0 0 0 1 0]
 [0 0 0 1 1]
 [0 0 0 1 2]
 [0 0 0 1 3]
 [0 0 0 1 4]
 [0 0 0 2 0]
 [0 0 0 2 1]
 [0 0 0 2 2]
 [0 0 0 2 3]
 [0 0 0 2 4]
 [0 0 0 3 0]
 [0 0 0 3 1]
 [0 0 0 3 2]
 [0 0 0 3 3]
 [0 0 0 3 4]
 [0 0 1 0 0]
 [0 0 1 0 1]
 [0 0 1 0 2]
 [0 0 1 0 3]
 [0 0 1 0 4]
 [0 0 1 1 0]
 [0 0 1 1 1]
 [0 0 1 1 2]
 [0 0 1 1 3]
 [0 0 1 1 4]
 [0 0 1 2 0]
 [0 0 1 2 1]
 [0 0 1 2 2]
 [0 0 1 2 3]
 [0 0 1 2 4]
 [0 0 1 3 0]
 [0 0 1 3 1]
 [0 0 1 3 2]
 [0 0 1 3 3]
 [0 0 1 3 4]
 [0 0 2 0 0]
 [0 0 2 0 1]
 [0 0 2 0 2]
 [0 0 2 0 3]
 [0 0 2 0 4]
 [0 0 2 1 0]
 [0 0 2 1 1]
 [0 0 2 1 2]
 [0 0 2 1 3]
 [0 0 2 1 4]
 [0 0 2 2 0]
 [0 0 2 2 1]
 [0 0 2 2 2]
 [0 0 2 2 3]
 [0 0 2 2 4]
 [0 0 2 3 0]
 [0 0 2 3 1]
 [0 0 2 3 2]
 [0 0 2 3 3]
 [0 0 2 3 4]
 [0 0 3 0 0]
 [0 0 3 0 1]
 [0 0 3 0 2]
 [0 0 3 0 3]
 [0 0 3 0 4]
 [0 0 3 1 0]
 [0 0 3 1 1]
 [0 0 3 1 2]
 [0 0 3 1 3]
 [0 0 3 1 4]
 [0 0 3 2 0]
 [0 0 3 2 1]
 [0 0 3 2 2]
 [0 0 3 2 3]
 [0 0 3 2